# Fetching Data from a SQL Database with Python

Before we can visualize the Kaggle survey data, we need to get it out of a PostgreSQL database and into a pandas DataFrame.

This notebook demonstrates **two approaches** for connecting Python to a SQL database:

1. **`psycopg2`**: a low-level PostgreSQL adapter (direct cursor-based queries)

2. **`SQLAlchemy`**: a higher-level abstraction that works with many database engines

Both approaches end the same way: we export the data to a CSV file so we can use it in the visualization notebook without re-running queries every time.

## Learning Objectives

1. Understand why we connect to databases from Python (vs. using a SQL client like DBeaver)

2. Create a database connection using `psycopg2` and `SQLAlchemy`

3. Execute a SQL query and load results into a pandas DataFrame

4. Export the DataFrame to a CSV for use in other notebooks


## Why Connect from Python?

A typical workflow without Python integration:

1. Clean data in Python → export to CSV

2. Import CSV into DBeaver → upload to database

3. Query in DBeaver → export results to CSV

4. Import CSV back into Python → visualize

That is four steps and two applications. Connecting Python directly to the database **cuts this to one step**.

## Approach 1: `psycopg2`

`psycopg2` is the most popular low-level PostgreSQL adapter for Python. It follows the DB-API 2.0 standard: you create a **connection**, then a **cursor**, then execute queries through the cursor.

In [1]:
# --- Imports for psycopg2 approach ---
import pandas as pd  # read SQL results into DataFrames
import psycopg2  # PostgreSQL adapter
import os
from dotenv import load_dotenv  # read credentials from .env file

print("Packages loaded!")

Packages loaded!


### Credentials from a `.env` file

Never hardcode database credentials in a notebook, they would end up on GitHub.

Instead, store them in a `.env` file (which is listed in `.gitignore`). The `dotenv` package reads this file and makes the values available as environment variables.

create a new file and call it `.env`.

Your `.env` file should look like:
```bash
 Copy this file to .env and fill in your own values. Do not commit .env.

# for psycopg2
# Replace USER_DB and PASSWORD with the username and password from your coaches (Dbeaver setup).
DATABASE = "postgres"
USER_DB = "<your-username>"
PASSWORD = "<your-password>"
HOST = "<database-host>"
PORT = "5432"

# for sqlalchemy
# Replace <your-username>, <your-password>, and <database-host> with the values from your coaches (Dbeaver setup).
DB_STRING=postgresql://<your-username>:<your-password>@<database-host>:5432/postgres
```
> You should already have the credentials in your pinned messages. 

In [2]:
# --- Load credentials from .env ---
# load_dotenv() reads the .env file and makes variables available via os.getenv()
load_dotenv()

DATABASE = os.getenv("postgres")
USER_DB = os.getenv("<your-username>")
PASSWORD = os.getenv("<your-password>")
HOST = os.getenv("<database-host>")
PORT = os.getenv("5432")

# Quick check (never print actual credentials!)
print(f"Database: {DATABASE}")
print(f"Host:     {HOST}")
print(f"Port:     {PORT}")

Database: None
Host:     None
Port:     None


In [4]:
import os

print(os.getcwd())

c:\Users\EMS-User\Desktop\ds-visualisationKSD


In [5]:
from dotenv import load_dotenv
import os

print(load_dotenv())

print(os.getenv("DATABASE"))
print(os.getenv("HOST"))

True
postgres
ds-sql-playground.c8g8r1deus2v.eu-central-1.rds.amazonaws.com


In [8]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

DATABASE = os.getenv("DATABASE")
USER_DB = os.getenv("USER_DB")
PASSWORD = os.getenv("PASSWORD")
HOST = os.getenv("HOST")
PORT = os.getenv("PORT")

print("DATABASE:", DATABASE)
print("HOST:", HOST)
print("PORT:", PORT)

DATABASE: postgres
HOST: ds-sql-playground.c8g8r1deus2v.eu-central-1.rds.amazonaws.com
PORT: 5432


In [9]:
conn = psycopg2.connect(
    database=DATABASE,
    user=USER_DB,
    password=PASSWORD,
    host=HOST,
    port=PORT
)

print("Connection established!")


Connection established!


In [7]:
conn = psycopg2.connect(
    database=DATABASE,
    user=USER_DB,
    password=PASSWORD,
    host=HOST,
    port=PORT
)

print("Connection established!")

OperationalError: connection to server at "localhost" (::1), port 5432 failed: Connection refused (0x0000274D/10061)
	Is the server running on that host and accepting TCP/IP connections?
connection to server at "localhost" (127.0.0.1), port 5432 failed: Connection refused (0x0000274D/10061)
	Is the server running on that host and accepting TCP/IP connections?


### Creating a Connection and Cursor

A **connection** opens a session with the database server.
A **cursor** is a Python object that lets you send SQL commands through that connection.

> **Tip**: If you get a connection error after correctly editing `.env`, restart your kernel, environment variables are only read when the kernel starts.

In [6]:
# --- Open connection to the database ---
conn = psycopg2.connect(
    database=DATABASE, user=USER_DB, password=PASSWORD, host=HOST, port=PORT
)

print("Connection established!")

OperationalError: connection to server at "localhost" (::1), port 5432 failed: Connection refused (0x0000274D/10061)
	Is the server running on that host and accepting TCP/IP connections?
connection to server at "localhost" (127.0.0.1), port 5432 failed: Connection refused (0x0000274D/10061)
	Is the server running on that host and accepting TCP/IP connections?


In [ ]:
# --- Create a cursor and run a test query ---
cur = conn.cursor()  # create cursor
cur.execute("SELECT * FROM datasets.kaggle_survey LIMIT 5")  # send SQL
rows = cur.fetchall()  # retrieve results

print(f"Retrieved {len(rows)} rows")
for row in rows:
    print(row)

In [ ]:
# --- Always close the connection when done ---
conn.close()
print("Connection closed.")

### Loading into a DataFrame with `pd.read_sql`

`pd.read_sql()` is a convenience wrapper, it sends a query and immediately returns a DataFrame. Much more useful than fetching raw tuples with a cursor.

In [ ]:
# --- Reconnect and load data directly into a DataFrame ---
conn = psycopg2.connect(
    database=DATABASE, user=USER_DB, password=PASSWORD, host=HOST, port=PORT
)

# --- Run a query and load results into a DataFrame ---
query = "SELECT * FROM datasets.kaggle_survey LIMIT 10"
df_psycopg = pd.read_sql(query, conn)

conn.close()
print(f"Shape: {df_psycopg.shape}")
df_psycopg.head()

## Approach 2: `SQLAlchemy`

`SQLAlchemy` uses a **connection string** (also called a database URL) that encodes all connection information in a single string. It supports many database engines (PostgreSQL, MySQL, SQLite, etc.) with the same API, very useful in production.

In [ ]:
# --- Import SQLAlchemy ---
from sqlalchemy import create_engine

# --- Load the connection string from .env ---
load_dotenv()
DB_STRING = os.getenv("DB_STRING")

# --- Create an engine from the connection string ---
# The engine is a factory that manages database connections
db = create_engine(DB_STRING)
print("Engine created!")

In [ ]:
# --- Load the full kaggle_survey table into a DataFrame ---
query = "SELECT * FROM datasets.kaggle_survey"
df_sqlalchemy = pd.read_sql(query, db)

print(f"Shape: {df_sqlalchemy.shape}")
df_sqlalchemy.head()

## Export to CSV

Now that we have the full dataset, we export it to a CSV file. This means we can load it instantly in the visualization notebook without re-running the SQL query every time.

In [ ]:
# --- Export to CSV ---
# index=False prevents pandas from writing the row numbers as a column
df_sqlalchemy.to_csv("data/kaggle_survey.csv", index=False)

print("Saved to data/kaggle_survey.csv")
print(
    f"File contains {len(df_sqlalchemy):,} rows and {len(df_sqlalchemy.columns)} columns"
)

## Checkpoint: SQL + Python

1. **What is the difference between a `connection` and a `cursor` in `psycopg2`?**

<details>
<summary>Show answer</summary>

A connection is the open link between your Python program and the database; it manages the session and transactions. A cursor is created from a connection and is the object you use to send a SQL statement and step through the rows that come back. In short, the connection is the pipe and the cursor is the tool you run queries with through it.
</details>

2. **Why do we store credentials in a `.env` file instead of directly in the notebook?**

<details>
<summary>Show answer</summary>

Keeping credentials in a `.env` file (which is git-ignored) keeps secrets like your username and password out of the notebook and out of version control, so you do not leak them when you push or share the file. It also lets each person use their own credentials without editing the code. The notebook reads the values at runtime with something like `os.getenv`.
</details>

3. **What does `pd.read_sql(query, conn)` return?**

<details>
<summary>Show answer</summary>

It runs the SQL query over the connection and returns the result as a pandas DataFrame, with one row per result row and columns named after the selected fields. That means you can immediately use the usual pandas tools (filtering, grouping, plotting) on the result without looping over rows by hand.
</details>

4. **What is the main advantage of `SQLAlchemy` over `psycopg2`?**
   *(Hint: think about working with different database engines)*

<details>
<summary>Show answer</summary>

SQLAlchemy is an abstraction layer that talks to many database engines (PostgreSQL, MySQL, SQLite, and others) through one consistent interface, so the same code can target a different database just by changing the connection string. `psycopg2` is a driver specific to PostgreSQL. That portability across engines is what the hint points at.
</details>

5. **Why do we export the data to a CSV rather than querying the database again in the visualization notebook?**

<details>
<summary>Show answer</summary>

Exporting once to a CSV lets the later visualization notebook load the data instantly from disk, with no database credentials, network connection, or running database required. It also freezes the exact dataset you analysed, so your results stay reproducible and you avoid hitting the database repeatedly.
</details>

> You are now ready to move on to **[4_Visualization_exercise](4_Visualization_exercise.ipynb)**!